# OncoResolve: Breast Cancer Subtyping and Prognosis Demo

This notebook showcases how to use the `OncoResolve` library to prepare transcriptomics data, predict PAM50 breast cancer subtypes using pre-trained machine learning models, and compute prognostic risk scores (Consensus Risk Scores, CRS).

### Modules Demonstrated:
1. **`utils`**: Harmonizing Entrez gene IDs, scaling, and feature alignment.
2. **`OncoClassifier`**: Running PAM50 subtyping using pre-trained SVM & LR models.
3. **`OncoPrognosis`**: Calculating consensus overall survival risk score.

In [1]:
import numpy as np
import pandas as pd
import joblib
import OncoResolve as orr
from pathlib import Path

print(f"OncoResolve version: {orr.__version__}")

OncoResolve version: 1.0.2


## Step 1: Generate Simulated Transcriptomics Data

To replicate a real bulk RNA-seq experiment, we will load the pre-trained model metadata and simulate log2-transformed expression profiles for a cohort of 50 patients across the required signature genes.

In [2]:
# Load required features from the survival model
prog = orr.OncoPrognosis()
required_features = sorted(prog.model_.params_.index.tolist())
print(f"Loaded {len(required_features)} model signature genes (e.g. {required_features[:5]}...)")

# Simulate raw expression values centered around typical TCGA log2 means
package_dir = Path(orr.__file__).resolve().parent
svm = joblib.load(package_dir / "models" / "final_probability_svm.pkl")
scaler = svm.named_steps["scaler"]

np.random.seed(42)
n_samples = 50
# Base expression centered around training set means
raw_data = np.random.normal(loc=scaler.mean_, scale=0.3 * scaler.scale_, size=(n_samples, len(required_features)))
raw_data = np.clip(raw_data, 0.0, None)  # Expression cannot be negative

barcodes = [f"PATIENT-{i:03d}" for i in range(n_samples)]
df_raw = pd.DataFrame(raw_data, index=barcodes, columns=required_features)

print(f"Generated raw dataset: {df_raw.shape[0]} patients, {df_raw.shape[1]} genes.")
df_raw.head(3)

Loaded 178 model signature genes (e.g. ['A2ML1', 'ABCA12', 'ABCC11', 'ABCC8', 'ACKR1']...)
Generated raw dataset: 50 patients, 178 genes.


,A2ML1,ABCA12,ABCC11,ABCC8,ACKR1,ADH1B,AFF3,AGR2,AGR3,AKR1B10,...,TP63,TPRG1,TRIM29,TRPV6,TTC36,UGT2B28,VGLL1,WIF1,WNK4,ZP2
PATIENT-000,7.943085,3.530150,5.175657,8.448124,11.269763,4.467906,8.331931,8.435953,6.759641,8.507388,...,3.456878,7.095512,8.983371,3.450458,6.689695,7.182097,6.567742,7.047345,5.268688,2.308002
PATIENT-001,7.031790,6.439182,5.152485,5.847239,10.414354,5.188783,6.481287,8.363799,7.651746,7.889096,...,2.002654,6.441612,9.125033,3.943002,6.940713,8.598893,7.163093,6.200013,5.235339,0.942283
PATIENT-002,7.326403,3.377093,4.833698,5.879940,12.040075,6.245305,6.599042,7.942104,7.857009,7.559156,...,2.830701,6.044089,8.372216,4.156853,6.907599,5.626315,6.938603,4.911077,5.186319,0.826351


## Step 2: Prepare & Scale the Data

Depending on the model, we require either raw expression data (for the subtyping classifiers which contain internal scalers) or Z-score cohort-scaled data (for prognosis and uniqueness modules).

In [3]:
# 1. Align features of raw data alphabetically (required for classifier)
df_aligned_raw = orr.align_features(df_raw, required_features)

# 2. Create scaled cohort (required for prognosis)
df_scaled = orr.scale_cohort(df_raw)
df_aligned_scaled = orr.align_features(df_scaled, required_features)

print("Data preparation complete!")

Data preparation complete!


## Step 3: Run Molecular Subtyping

We will load the pre-trained `OncoClassifier` (RBF-SVM) to predict the PAM50 molecular subtypes ('basal', 'her2', 'luminal_A', 'luminal_B', 'normal') and their probabilities.

In [4]:
# Initialize pre-trained SVM classifier
clf = orr.OncoClassifier(model_type="svm")

# Run subtyping on raw aligned features
predictions = clf.predict(df_aligned_raw)
probabilities = clf.predict_proba(df_aligned_raw)

# Display predictions
subtyping_results = pd.DataFrame({
    "Predicted_Subtype": predictions
}, index=df_raw.index).join(probabilities)

print("PAM50 Subtyping Summary:")
print(subtyping_results["Predicted_Subtype"].value_counts())
subtyping_results.head(5)

PAM50 Subtyping Summary:
Predicted_Subtype
luminal_A    33
luminal_B    17
Name: count, dtype: int64


,Predicted_Subtype,basal,her2,luminal_A,luminal_B,normal
PATIENT-000,luminal_B,NaN,NaN,NaN,NaN,NaN
PATIENT-001,luminal_A,NaN,NaN,NaN,NaN,NaN
PATIENT-002,luminal_B,NaN,NaN,NaN,NaN,NaN
PATIENT-003,luminal_A,NaN,NaN,NaN,NaN,NaN
PATIENT-004,luminal_A,NaN,NaN,NaN,NaN,NaN


## Step 4: Predict Prognostic Risk Scores (CRS)

Now we calculate the overall survival Consensus Risk Score (CRS) for each patient using the regularized Ridge Cox model. Since it expects scaled data, we pass `df_aligned_scaled`.

In [5]:
# Initialize pre-trained prognosis model
prog = orr.OncoPrognosis()

# Predict risk scores
crs_scores = prog.predict_risk(df_aligned_scaled)

# Merge predictions
final_cohort_profile = subtyping_results.copy()
final_cohort_profile["Consensus_Risk_Score"] = crs_scores

print("Top 5 High-Risk Patients:")
final_cohort_profile.sort_values(by="Consensus_Risk_Score", ascending=False).head(5)

Top 5 High-Risk Patients:


,Predicted_Subtype,basal,her2,luminal_A,luminal_B,normal,Consensus_Risk_Score
PATIENT-042,luminal_B,NaN,NaN,NaN,NaN,NaN,6.649124
PATIENT-041,luminal_A,NaN,NaN,NaN,NaN,NaN,6.010141
PATIENT-032,luminal_B,NaN,NaN,NaN,NaN,NaN,5.995531
PATIENT-010,luminal_B,NaN,NaN,NaN,NaN,NaN,5.977616
PATIENT-011,luminal_B,NaN,NaN,NaN,NaN,NaN,5.922468
